# Фаза 6: подготовка рабочей копии и отдельный запуск агента

Обновлено 17.10.2026. Ноутбук не запускает ИИ при обычной подготовке и не читает каналы осциллограмм. Сначала настраиваешь **копию** своей файловой базы; потом отдельно включаешь API-пилот. Составление 17 наших черновиков уже закончено; этот ноутбук готовит повторяемую обработку других/тех же входов.

Порядок: 1) пути и годы → 2) детерминированная подготовка → 3) просмотр сводки/ошибок → 4) контексты агента → 5) один сетевой прогон → 6) сохранение итогов вне контекста → 7) при необходимости пакет/карта безымянных файлов.


## 1. Пути, режимы и поля

Ниже реализован режим «не указан выход — работаем в той же папке».

| Поле | Что означает / что сохраняется |
|---|---|
| `PROJECT` | Каталог репозитория с кодом osc_tools. Сам по себе не является обрабатываемой базой |
| `SOURCE_ROOT` | Твоя рабочая **копия** корня, внутри которой есть годы `2010`, `2024`, `2025` и т. п. Нужен настоящий путь; пример ниже специально не создаёт/не копирует базу автоматически |
| `OUTPUT_ROOT=None` | Подготовка **на месте**: обычные файлы остаются на своих местах, `_phase6` в корне хранит индексы/отметки, а `_phase6_extracted` в папках — выбранные архивные вложения. Повторный проход пропускает готовые неизменившиеся папки |
| `OUTPUT_ROOT=Path(...)` | Отдельный подготовленный снимок вне SOURCE_ROOT. Копируются выбранные файлы, используются blob-дубли. Существующий законченный снимок можно повторно открыть; для нового состава создаётся новый путь. В этом режиме новая папка нужна только для нового снимка, не для каждого запуска ячейки |
| `BLOB_ROOT` | Только для отдельного снимка: общее хранилище SHA256 нескольких снимков. При None — внутри снимка. В режиме на месте не используется |
| `YEARS` | Список строк, например `["2025"]`; `[]` — все найденные годы. Другие годы не подготавливаются |
| `SEED` | Реестр постоянных ID. Готовый case_registry.json подходит к нашей демонстрационной выборке с теми же относительными путями. Для другой базы — None или её собственный реестр |
| `EXTRACT_ARCHIVES` | Перечислить архивы и извлекать выбранные документальные/технические типы. False — только список, вложения не попадут к агенту |
| `MAX_MEMBER_MB`, `MAX_TOTAL_MB`, `MAX_DEPTH` | Лимиты на один элемент, суммарные байты извлечения за проход и вложенность архивов. Превышения явно записываются; это не основание удалять оригинал |
| `VERIFY_UNCHANGED_HASHES` | На месте: False — быстро проверить состав, размер и mtime. True — дополнительно перечитать SHA256 перед пропуском. Совпадение только размера/mtime не гарантирует неизменность при их намеренном восстановлении |
| `WRITE_SOURCE_INDEXES` | В отдельном режиме дополнительно писать маленькие индексы в исходные папки. На месте свои отметки `_phase6/prepared.json` создаются автоматически |
| `MODEL`, `RUN_REMOTE` | Используются позже; пока RUN_REMOTE=False, сетевых запросов и расходов API нет |

На месте не удаляются нулевые файлы, оригинальные архивы, записи и документы. Нулевые файлы/кэши исключаются из выбранного входа агента. Только одиночные справки корня года копируются в отдельную маленькую рабочую группу, чтобы не дать агенту доступ сразу ко всему году.


In [ ]:
from pathlib import Path
import sys, json, os

PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "osc_tools/corpus").is_dir()), None)
if PROJECT is None:
    raise RuntimeError("Открой ноутбук внутри репозитория либо задай PROJECT вручную")
sys.path.insert(0, str(PROJECT))
from osc_tools.corpus.preparation import PreparationConfig, CorpusPreparer, prepare_agents
from osc_tools.corpus.remote_agent import run_remote, run_batch
from osc_tools.corpus.report_export import export_reports
from osc_tools.corpus.record_identity import register_derivative, match_flat, restore_flat
from osc_tools.corpus.agent_runtime import write_json

SOURCE_ROOT = PROJECT / "private/phase6/input_copy"  # ИЗМЕНИ: путь к твоей копии базы; сейчас пример не существует
OUTPUT_ROOT = None  # На месте. Для отдельного снимка: PROJECT / "private/phase6/ai_temp/prepared_001"
BLOB_ROOT = None  # Для отдельного режима можно указать общий каталог вне SOURCE_ROOT
YEARS = []
SEED = PROJECT / "docs/phase_discription/phase_6/case_registry.json"  # другая база: None
DOCS = PROJECT / "docs/phase_discription/phase_6"
EXTRACT_ARCHIVES = True
MAX_MEMBER_MB, MAX_TOTAL_MB, MAX_DEPTH = 128, 512, 1
VERIFY_UNCHANGED_HASHES = False
WRITE_SOURCE_INDEXES = False
# os.environ["PHASE6_7ZIP"] = r"C:\Program Files\7-Zip\7z.exe"  # если 7z/7zz не в PATH
print({"source": str(SOURCE_ROOT), "years": YEARS, "mode": "на месте" if OUTPUT_ROOT is None else "отдельный снимок"})


## 2. Выполнить подготовку без ИИ

Эта ячейка обходит выбранные годы, закрепляет ID, считает SHA256/MD5, связывает известные компоненты записей по именам, учитывает пустые файлы и выбирает вложения архивов. Каналы, физические процессы и причины аварий она не анализирует. EXE/SFX не запускаются; архивный отбор по техническому типу не является смысловой оценкой важности.

На месте есть отметка `_phase6/cases/<ID>/prepared.json`: неизменившиеся готовые папки пропускаются; удалённое извлечение или изменённый исходник инициирует обработку этой папки заново. Наличие одного случайного файла report.md само по себе не считается готовностью. Неполный проход/ошибка архива не получает готовую отметку.

Зависимости: pypdf для текстового слоя PDF, Pillow для BMP/TIFF, 7-Zip для RAR/7z/старого DOC, pdftoppm для страниц PDF. Они автоматически не устанавливаются.


In [ ]:
if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError("Задай SOURCE_ROOT — существующую рабочую копию базы")
config = PreparationConfig(source_root=str(SOURCE_ROOT), output_root=str(OUTPUT_ROOT) if OUTPUT_ROOT is not None else None,
    seed_inventory=str(SEED) if SEED else None, years=tuple(YEARS), extract_archives=EXTRACT_ARCHIVES,
    max_member_bytes=MAX_MEMBER_MB * 1024**2, max_total_extracted_bytes=MAX_TOTAL_MB * 1024**2,
    max_archive_depth=MAX_DEPTH, seven_zip=os.environ.get("PHASE6_7ZIP"),
    blob_root=str(BLOB_ROOT) if BLOB_ROOT else None, write_source_indexes=WRITE_SOURCE_INDEXES,
    verify_unchanged_hashes=VERIFY_UNCHANGED_HASHES)
preparer = CorpusPreparer(config)
PREPARED_ROOT = preparer.output
if not preparer.in_place and (PREPARED_ROOT / "summary.json").exists():
    from dataclasses import asdict
    expected = json.loads(json.dumps(asdict(config)))
    saved = json.loads((PREPARED_ROOT / "preparation_config.json").read_text(encoding="utf-8"))
    if expected != saved:
        raise ValueError("Параметры отдельного снимка изменились: новый OUTPUT_ROOT или режим на месте")
    summary = json.loads((PREPARED_ROOT / "summary.json").read_text(encoding="utf-8"))
    print("Открыт законченный отдельный снимок. Изменения первичного корня в него автоматически не добавляются.")
else:
    summary = preparer.run()
display(summary)


## 3. Проверить, что получилось

`cases_processed`/`cases_skipped` относятся к режиму на месте; повтор без изменений должен показать пропуск. `source_files_unchanged` относится к первичным байтам, а новые служебные каталоги/извлечения ожидаемо появляются. `extracted_bytes` — байты извлечения за проход; это не объём отправленного API.

Таблицы ниже: исходная папка ↔ постоянный ID; состояние каждого архива и его проблемных элементов; исключённые нулевые файлы/кэши. Для `partial`, `archive_failed/listing_failed`, шифрования или лимита сначала посмотри причину. 17 комплектов не означает 17 осциллограмм: папка может содержать несколько эпизодов, записи или только документ.


In [ ]:
provenance = json.loads((PREPARED_ROOT / "provenance_index.json").read_text(encoding="utf-8"))
archives = json.loads((PREPARED_ROOT / "archive_manifest.json").read_text(encoding="utf-8"))
display([{"case_id":c["case_id"], "source_folder":c["source_folder"], "input":c["input_root"], "files":len(c["files"])} for c in provenance["cases"]])
display([{"archive":a["path"], "status":a["status"], "issues":[m for m in a["members"] if m["status"] not in
          {"extracted","directory","listed_not_read","nested_archive_processed"}]} for a in archives])
display([{"case_id":c["case_id"],"file":f["logical_path"],"status":f["preparation_status"]}
         for c in provenance["cases"] for f in c["files"] if not f.get("prepared_path")])


## 4. Подготовить отдельные контексты агента

Это ещё не обращение к модели. В каждый контекст автоматически входят полные инструкция и словарь, манифест выбранной папки и пустое состояние. Доступа к другим комплектам и нашим отчётам через инструменты нет. На месте контексты — `_phase6/agent_runs/<AGENT_RUN>/<case-ID>`; их нельзя удалять, пока незавершённый прогон нужен для продолжения.

Для нашей выборки case_000005=G06. Для своей базы выбери ID по таблице выше. CASE_IDS=None подготовит все контексты. Повтор того же AGENT_RUN открывает существующее только при неизменных входах/инструкции/словаре; для трёх независимых прогонов нужны api_001/api_002/api_003. Прежние имена файлов отчётов из нашей подборки эта операция не меняет.


In [ ]:
CASE_IDS = ["case_000005"]  # первая проба G06; другой корпус — выбрать из таблицы; None — все
AGENT_RUN = "api_001"
contexts = prepare_agents(PREPARED_ROOT, DOCS / "PHASE_6_EXTRACTION_PROMPT_DRAFT.md",
    DOCS / "PHASE_6_TERMINOLOGY_DRAFT.md", case_ids=CASE_IDS, run_name=AGENT_RUN, reuse_existing=True)
labels = {c["case_id"]:c["source_folder"] for c in provenance["cases"]}
display([{**c,"source_folder":labels[c["case_id"]]} for c in contexts])


## 5. Один удалённый прогон и сохранение отчёта

Задай точное имя доступной модели и OPENAI_API_KEY в окружении Jupyter, не в файле ноутбука. Только RUN_REMOTE=True передаёт выбранные тексты/картинки провайдеру и расходует токены. Сырые архивы и сигналы целиком не загружаются. RESUME=True — продолжить остановленный контекст, а не новый независимый результат.

По завершении `export_reports` сохраняет отчёт, карту качества, состав источников и краткие параметры прогона в `<PREPARED_ROOT>/reports/<AGENT_RUN>/<год>/<исходная папка>/revision_.../`. Это постоянный результат отдельно от рабочего контекста. Первичный архив/извлечение и неэкспортированный контекст пока сохраняй. Статусы лимита/ошибки или наличия черновика не подтверждают правильность выводов.


In [ ]:
MODEL = ""
RUN_REMOTE = False
RESUME = False
if RUN_REMOTE:
    if not MODEL or not os.environ.get("OPENAI_API_KEY"):
        raise ValueError("Укажи MODEL и OPENAI_API_KEY в окружении процесса")
    display(run_remote(contexts[0]["workspace"], MODEL, max_requests=80, resume=RESUME))
    display(export_reports(PREPARED_ROOT, contexts[:1], AGENT_RUN))
else:
    print("API выключен: выполнена только локальная подготовка.")


## 6. Пакет после проверки одного результата

При CASE_IDS=None контексты подготовлены для всех выбранных папок. Пакет идёт от новых годов к старым; завершённые контексты пропускает. Результаты экспортируются после пакета, чтобы не оставить единственную копию отчёта в рабочем состоянии. Общего денежного лимита здесь нет: установлен лимит запросов на комплект, usage сохраняется отдельно. Сначала оцени один реальный прогон.


In [ ]:
RUN_BATCH = False
if RUN_BATCH:
    if not MODEL or not os.environ.get("OPENAI_API_KEY"):
        raise ValueError("Нужны MODEL и ключ в окружении")
    batch_results = run_batch([c["workspace"] for c in contexts], MODEL, resume=RESUME, max_requests=80)
    write_json(PREPARED_ROOT / (AGENT_RUN + "__batch.json"), batch_results)
    display(batch_results)
    display(export_reports(PREPARED_ROOT, contexts, AGENT_RUN))


## 7. Карта записей и безымянный набор — необязательная ветка

record_index связывает папку, исходные имена, SHA256 компонентов и прежний MD5. DEX — контейнер без чтения внутренних записей. Связь файлов по имени/хешу не доказывает тип аварии. Ниже только просмотр индекса.

Когда отдельный обработчик преобразует запись, сразу зарегистрируй новые файлы и параметры через register_derivative. После изменения байтов прежний хеш не узнаёт результат самостоятельно.

Повтор подготовки на месте сохраняет зарегистрированные производные. `current_input=False` обозначает прежнюю версию записи или запись вне выбранных сейчас годов; такие соответствия не теряются.


In [ ]:
RECORD_INDEX = PREPARED_ROOT / "record_index.json"
records = json.loads(RECORD_INDEX.read_text(encoding="utf-8"))
display([{"record_id":r["record_id"],"case_id":r["case_id"],"source_folder":r["source_folder"],"kind":r["kind"],"current_input":r.get("current_input",True),
          "files":[f["logical_path"] for f in r["components"]]} for r in records["records"]])
# register_derivative(RECORD_INDEX, record_id, [new_cfg, new_dat], "conversion", {"crop":False})


Укажи FLAT_ROOT только при наличии безымянного набора. Сначала создаётся план: matched — однозначно, unknown — неизвестно, multiple_matches_review — несколько возможных соответствий. Копирование включается отдельно и выполняется в новый RESTORE_ROOT вне обоих входных каталогов. Ничего не перемещается из исходной базы и не затирается.


In [ ]:
FLAT_ROOT = None
RESTORE_ROOT = None
APPLY_RESTORE = False
if FLAT_ROOT is not None:
    return_plan = match_flat(RECORD_INDEX, FLAT_ROOT)
    write_json(PREPARED_ROOT / "flat_return_plan.json", return_plan)
    display(return_plan["files"])
    if APPLY_RESTORE:
        if RESTORE_ROOT is None:
            raise ValueError("Укажи новый RESTORE_ROOT")
        display(restore_flat(return_plan, RESTORE_ROOT))


## Где что искать и что можно очищать

На месте всё внутри указанной копии корпуса: `_phase6/cases` — отметки/происхождение, `_phase6/record_index.json` — общий индекс, `<папка>/_phase6_extracted` — воспроизводимые вложения, `_phase6/agent_runs` — рабочие контексты и ещё не экспортированные отчёты, `_phase6/reports` — сохранённые результаты по настоящим именам. Извлечения можно удалить для освобождения места, следующий проход восстановит их из сохранённых архивов. Рабочие контексты удаляй после экспорта или когда намеренно отказываешься от продолжения. Готовые отчёты и первичные архивы сохраняй.

Наша проверяемая подборка — `private/phase6/reports`, вход через `private/phase6/README.md`. Там 17 черновиков; это завершённый охват текущей выборки, а не приёмка специалистом. Замечания для четырёх исходных случаев — в docs/phase_discription/phase_6/reports/README.md.
